# Getting basic dataset descriptors

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
ENGLISH_DATA_PATH = 'english/data/ckpts/rosen'
XLING_DATA_PATH = 'xling/data/ckpts/rosen'

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(ENGLISH_DATA_PATH) + grab_all_files(XLING_DATA_PATH)
len(fs)

1695

## Processing individual datasets

In [5]:
data, needs_avar = [], []

In [6]:
for f in tqdm(fs):
    table = pq.ParquetFile(f)
    df = table.read(columns=['corpus', 'conversation_id', 'x_speaker']).to_pandas()

    if 'AVAR' not in table.schema.names:
        needs_avar += [f]

    for corpus in df['corpus'].unique():
        sub = df.loc[df['corpus'].isin([corpus])]
        data += [
            {
                'xling': 'xling-' in f,
                'corpus': corpus,
                'conversations': sub['conversation_id'].nunique(),
                'speakers': sub['x_speaker'].nunique(),
                'comparisons': len(sub),
            }
        ]

data = pd.DataFrame(data)

  0%|          | 0/1695 [00:00<?, ?it/s]

In [7]:
needs_avar

[]

In [8]:
data['corpus'] = ['CANDOR' if ('CANDOR' in it) else it for it in data['corpus']]
data['corpus'] = ['MICASE' if ('MICASE' in it) else it for it in data['corpus']]

In [9]:
data = data.groupby(by=['xling', 'corpus']).agg('sum').reset_index()

In [10]:
data.head(n=30)

,xling,corpus,conversations,speakers,comparisons
0,False,CABNC,1998,6130,43167739
1,False,CANDOR,1656,3312,89866650
2,False,CORAAL,271,588,56663656
3,False,DISPEL,19,38,485633
4,False,Frederiksen,2,4,3709
5,False,GCSAusE,40,106,324155
6,False,Graesser,3,6,52652
7,False,MICASE,152,1756,41520784
8,False,SBCSAE,60,340,12398227
9,False,SCoSE,32,87,2770221


In [11]:
data['conversations'].sum(), data['speakers'].sum(), data['comparisons'].sum()

(np.int64(5812), np.int64(16025), np.int64(372256988))

English data characteristics

In [12]:
data['conversations'].loc[~data['xling']].sum(), data['speakers'].loc[~data['xling']].sum(), data['comparisons'].loc[~data['xling']].sum()

(np.int64(4522), np.int64(12996), np.int64(269146433))

In [13]:
data[[col for col in list(data) if col not in ['xling']]].loc[~data['xling']].to_csv('english-data-stats.csv', index=False, encoding='utf8')

Xling data characteristics

In [14]:
data['conversations'].loc[data['xling']].sum(), data['speakers'].loc[data['xling']].sum(), data['comparisons'].loc[data['xling']].sum()

(np.int64(1290), np.int64(3029), np.int64(103110555))

In [15]:
data[[col for col in list(data) if col not in ['xling']]].loc[data['xling']].to_csv('xling-data-stats.csv', index=False, encoding='utf8')